In [1]:
# NativeNext management setup; original course cells follow with the explicitly documented batch-parameter override.
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261024T041056-14302642')
import next_identity as _identity
_identity.write_child_runtime_identity()
import next_adapter as _next
_next.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261024T041056-14302642\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261024T041056-14302642','04_fastai_part2/17_DDPM_v2.ipynb')


# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original numbered course notebook, adapted for Colab. Run in numerical order: later lessons load models trained by earlier lessons. Use the same Drive result folder across sessions. W&B logs stay offline. Full training can exceed a free Colab session; QUICK_RUN does not shorten these original Part 2 experiments.

Source: https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/17_DDPM_v2.ipynb

In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '04_fastai_part2', 'part2', '17_DDPM_v2.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hi\20261024T041056-14302642\source\HaiLong_Colab_Exercises\04_fastai_part2
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hi\20261024T041056-14302642\source\HaiLong_Colab_Exercises\04_fastai_part2\results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Retain original datasets, model definitions, training schedules and instructional comparisons.
- Run prerequisite checkpoint-producing notebooks first; do not replace missing weights with untrained models.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [3]:
MAX_BATCH_SIZE = 512  #@param {type:"integer"}
# Increase on a larger GPU to match the original batch sizes.
assert MAX_BATCH_SIZE > 0

In [4]:
#|default_exp accel

# Denoising Diffusion Probabilistic Models with miniai

## Imports

In [5]:
#|export
import pickle,gzip,math,os,time,shutil,torch,random,logging
import fastcore.all as fc,matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
from collections.abc import Mapping
from pathlib import Path
from functools import partial

from fastcore.foundation import L
import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torch.optim import lr_scheduler

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *
from miniai.sgd import *
from miniai.resnet import *
from miniai.augment import *

In [6]:
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset,load_dataset_builder

mpl.rcParams['image.cmap'] = 'gray_r'
logging.disable(logging.WARNING)

## Load the dataset

In [7]:
xl,yl = 'image','label'
name = "zalando-datasets/fashion_mnist"
dsd = load_dataset(name)

@inplace
def transformi(b): b[xl] = [F.pad(TF.to_tensor(o), (2,2,2,2)) for o in b[xl]]

bs = min(128, MAX_BATCH_SIZE)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=8)

README.md: 0.00B [00:00, ?B/s]

C:\Users\[LOCAL_USER]\.v\hlimg31\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hi\20261024T041056-14302642\private_cache\hf\hub\datasets--zalando-datasets--fashion_mnist. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/30.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/5.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [8]:
dt = dls.train
xb,yb = next(iter(dt))

In [9]:
show_images(xb[:16], imsize=1.5)

In [10]:
betamin,betamax,n_steps = 0.0001,0.02,1000
beta = torch.linspace(betamin, betamax, n_steps)
alpha = 1.-beta
alphabar = alpha.cumprod(dim=0)
sigma = beta.sqrt()

In [11]:
plt.plot(beta);

In [12]:
plt.plot(sigma);

In [13]:
plt.plot(alphabar);

In [14]:
def noisify(x0, ᾱ):
    device = x0.device
    n = len(x0)
    t = torch.randint(0, n_steps, (n,), dtype=torch.long)
    ε = torch.randn(x0.shape, device=device)
    ᾱ_t = ᾱ[t].reshape(-1, 1, 1, 1).to(device)
    xt = ᾱ_t.sqrt()*x0 + (1-ᾱ_t).sqrt()*ε
    return (xt, t.to(device)), ε

In [15]:
(xt,t),ε = noisify(xb[:25],alphabar)
t

tensor([530, 329, 850, 697, 967, 659, 634, 760, 827, 480, 340, 181, 398, 339,
        963, 789, 389, 401, 750, 337, 526, 321, 572, 967, 947])

In [16]:
titles = fc.map_ex(t, '{}')
show_images(xt, imsize=1.5, titles=titles)

## Training

In [17]:
from diffusers import UNet2DModel

In [18]:
@torch.no_grad()
def sample(model, sz, alpha, alphabar, sigma, n_steps):
    device = next(model.parameters()).device
    x_t = torch.randn(sz, device=device)
    preds = []
    for t in reversed(range(n_steps)):
        t_batch = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)
        z = (torch.randn(x_t.shape) if t > 0 else torch.zeros(x_t.shape)).to(device)
        ᾱ_t1 = alphabar[t-1]  if t > 0 else torch.tensor(1)
        b̄_t = 1 - alphabar[t]
        b̄_t1 = 1 - ᾱ_t1
        x_0_hat = ((x_t - b̄_t.sqrt() * learn.model((x_t, t_batch)))/alphabar[t].sqrt()).clamp(-1,1)
        x_t = x_0_hat * ᾱ_t1.sqrt()*(1-alpha[t])/b̄_t + x_t * alpha[t].sqrt()*b̄_t1/b̄_t + sigma[t]*z
        preds.append(x_t.cpu())
    return preds

In [19]:
class DDPMCB(Callback):
    order = DeviceCB.order+1
    def __init__(self, n_steps, beta_min, beta_max):
        super().__init__()
        fc.store_attr()
        self.beta = torch.linspace(self.beta_min, self.beta_max, self.n_steps)
        self.α = 1. - self.beta 
        self.ᾱ = torch.cumprod(self.α, dim=0)
        self.σ = self.beta.sqrt()
    
    def before_batch(self, learn): learn.batch = noisify(learn.batch[0], self.ᾱ)
    def sample(self, model, sz): return sample(model, sz, self.α, self.ᾱ, self.σ, self.n_steps)

In [20]:
class UNet(UNet2DModel):
    def forward(self, x): return super().forward(*x).sample

In [21]:
ddpm_cb = DDPMCB(n_steps=1000, beta_min=0.0001, beta_max=0.02)

In [22]:
model = UNet(in_channels=1, out_channels=1, block_out_channels=(16, 32, 64, 64), norm_num_groups=8)

learn = TrainLearner(model, dls, nn.MSELoss())
learn.fit(train=False, cbs=[ddpm_cb,SingleBatchCB()])
(xt,t),ε = learn.batch
show_images(xt[:25], titles=fc.map_ex(t[:25], '{}'), imsize=1.5)

In [23]:
lr = 5e-3
epochs = 5

tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [ddpm_cb, DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched)]

In [24]:
model = UNet(in_channels=1, out_channels=1, block_out_channels=(16, 32, 64, 128), norm_num_groups=8)

In [25]:
def init_ddpm(model):
    for o in model.down_blocks:
        for p in o.resnets:
            p.conv2.weight.data.zero_()
            for p in fc.L(o.downsamplers): init.orthogonal_(p.conv.weight)

    for o in model.up_blocks:
        for p in o.resnets: p.conv2.weight.data.zero_()

    model.conv_out.weight.data.zero_()

In [26]:
init_ddpm(model)

In [27]:
opt_func = partial(optim.Adam, eps=1e-5)

In [28]:
learn = TrainLearner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [29]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.225 GB; runtime free 406.454 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.024 GB; runtime free 406.454 GB. Budget is not Drive quota.


loss,epoch,train
0.147,0,train
0.026,0,eval
0.023,1,train
0.021,1,eval
0.020,2,train
0.019,2,eval
0.019,3,train
0.019,3,eval
0.018,4,train
0.018,4,eval


Storage check (checkpoint_write): artifacts 0.016 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 406.401 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.049 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 406.367 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.098 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.985 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.098 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.984 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.098 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.989 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.098 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.988 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.098 GB; configured budget 80.000 GB; additional estimate 0.024 GB; runtime free 405.988 GB. Budget is not Drive quota.


In [30]:
mdl_path = Path('models')

In [31]:
torch.save(learn.model, mdl_path/'fashion_ddpm2.pkl')

In [32]:
learn.model = torch.load(mdl_path/'fashion_ddpm2.pkl', weights_only=False)

## Sampling

In [33]:
samples = ddpm_cb.sample(learn.model, (16, 1, 32, 32))

In [34]:
show_images(samples[-1], figsize=(5,5))

## Mixed Precision

In [35]:
bs = min(512, MAX_BATCH_SIZE)

In [36]:
next(iter(DataLoader(tds['train'], batch_size=2)))

{'image': tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]]]),
 'label': tensor([9, 0])}

In [37]:
def collate_ddpm(b): return noisify(default_collate(b)[xl], alphabar)
def dl_ddpm(ds): return DataLoader(ds, batch_size=bs, collate_fn=collate_ddpm, num_workers=4)

In [38]:
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

In [39]:
#|export
class MixedPrecision(TrainCB):
    order = DeviceCB.order+10
    
    def before_fit(self, learn): self.scaler = torch.cuda.amp.GradScaler()

    def before_batch(self, learn):
        self.autocast = torch.autocast("cuda", dtype=torch.float16)
        self.autocast.__enter__()

    def after_loss(self, learn): self.autocast.__exit__(None, None, None)
        
    def backward(self, learn): self.scaler.scale(learn.loss).backward()

    def step(self, learn):
        self.scaler.step(learn.opt)
        self.scaler.update()

In [40]:
lr = 1e-2
epochs = 8
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), MixedPrecision(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched)]
model = UNet(in_channels=1, out_channels=1, block_out_channels=(16, 32, 64, 128), norm_num_groups=8)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [41]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.033 GB; configured budget 80.000 GB; additional estimate 0.225 GB; runtime free 405.929 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.033 GB; configured budget 80.000 GB; additional estimate 0.024 GB; runtime free 405.929 GB. Budget is not Drive quota.


C:\Users\[LOCAL_USER]\.v\hi\20261024T041056-14302642\private_cache\temp\ipykernel_57844\2894292263.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  def before_fit(self, learn): self.scaler = torch.cuda.amp.GradScaler()


loss,epoch,train
0.262,0,train
0.033,0,eval
0.029,1,train
0.031,1,eval
0.026,2,train
0.025,2,eval
0.022,3,train
0.021,3,eval
0.020,4,train
0.019,4,eval


Storage check (checkpoint_write): artifacts 0.050 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.913 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

...

Storage check (checkpoint_write): artifacts 0.132 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.771 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.132 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.770 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.132 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.762 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.132 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.762 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.132 GB; configured budget 80.000 GB; additional estimate 0.024 GB; runtime free 405.762 GB. Budget is not Drive quota.


In [42]:
samples = sample(learn.model, (32, 1, 32, 32), alpha, alphabar, sigma, n_steps)

In [43]:
show_images(samples[-1][:25], imsize=1.5)

In [44]:
torch.save(learn.model, 'models/fashion_ddpm_mp.pkl')

## Accelerate

`pip install accelerate` before running this section.

In [45]:
#|export
from accelerate import Accelerator

In [46]:
#|export
class AccelerateCB(TrainCB):
    order = DeviceCB.order+10
    def __init__(self, n_inp=1, mixed_precision="fp16"):
        super().__init__(n_inp=n_inp)
        self.acc = Accelerator(mixed_precision=mixed_precision)
        
    def before_fit(self, learn):
        learn.model,learn.opt,learn.dls.train,learn.dls.valid = self.acc.prepare(
            learn.model, learn.opt, learn.dls.train, learn.dls.valid)

    def backward(self, learn): self.acc.backward(learn.loss)

In [47]:
def noisify(x0, ᾱ):
    device = x0.device
    n = len(x0)
    t = torch.randint(0, n_steps, (n,), dtype=torch.long)
    ε = torch.randn(x0.shape, device=device)
    ᾱ_t = ᾱ[t].reshape(-1, 1, 1, 1).to(device)
    xt = ᾱ_t.sqrt()*x0 + (1-ᾱ_t).sqrt()*ε
    return xt, t.to(device), ε

In [48]:
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

In [49]:
class DDPMCB2(Callback):
    def after_predict(self, learn): learn.preds = learn.preds.sample

In [50]:
model = UNet2DModel(in_channels=1, out_channels=1, block_out_channels=(16, 32, 64, 128), norm_num_groups=8)
init_ddpm(model)
cbs = [DDPMCB2(), DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), AccelerateCB(n_inp=2)]
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [51]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.067 GB; configured budget 80.000 GB; additional estimate 0.225 GB; runtime free 405.640 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.067 GB; configured budget 80.000 GB; additional estimate 0.024 GB; runtime free 405.640 GB. Budget is not Drive quota.


loss,epoch,train
0.267,0,train
0.034,0,eval
0.029,1,train
0.028,1,eval
0.025,2,train
0.024,2,eval
0.022,3,train
0.021,3,eval
0.020,4,train
0.020,4,eval


Storage check (checkpoint_write): artifacts 0.084 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.557 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.116 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 405.532 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.165 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 404.316 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.165 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 404.316 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.166 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 404.314 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.166 GB; configured budget 80.000 GB; additional estimate 0.056 GB; runtime free 404.314 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.166 GB; configured budget 80.000 GB; additional estimate 0.024 GB; runtime free 404.314 GB. Budget is not Drive quota.


## A sneaky trick

In [52]:
class MultDL:
    def __init__(self, dl, mult=2): self.dl,self.mult = dl,mult
    def __len__(self): return len(self.dl)*self.mult
    def __iter__(self):
        for o in self.dl:
            for i in range(self.mult): yield o

In [53]:
dls.train = MultDL(dls.train)

## Export -

In [54]:
# The bundled miniai package is ready to import; no export is required.

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [55]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [56]:
# NativeNext independent numerical probe.
_next.probe(globals())


__HAILONG_NATIVE_NEXT_FULL_EVIDENCE__{"contract": {"passed": true, "ordinary_fits": 3, "ordinary_epochs": 21, "ordinary_train_batches": 4233, "ordinary_valid_batches": 520, "actual_adam_optimizer_updates": 4232, "amp_skips_are_not_updates": true, "sampler_steps": [1000, 1000], "sampler_images": [16, 32], "whole_model_pickles": 2}, "full_course_probe_verified": true, "full_dataset_pixels_and_cache_reverified": true, "state_roundtrips": [{"cell_index": 33, "strict_model_optimizer_gradient_rng_callback_state_restored": true, "actual_onecycle_curve_and_skips_verified": true, "cuda_rng_validation": "Saved ByteTensor structure only; no CUDA generator replay claimed."}, {"cell_index": 47, "strict_model_optimizer_gradient_rng_callback_state_restored": true, "actual_onecycle_curve_and_skips_verified": true, "cuda_rng_validation": "Saved ByteTensor structure only; no CUDA generator replay claimed."}, {"cell_index": 59, "strict_model_optimizer_gradient_rng_callback_state_restored": true, "actual_

{'contract': {'passed': True,
  'ordinary_fits': 3,
  'ordinary_epochs': 21,
  'ordinary_train_batches': 4233,
  'ordinary_valid_batches': 520,
  'actual_adam_optimizer_updates': 4232,
  'amp_skips_are_not_updates': True,
  'sampler_steps': [1000, 1000],
  'sampler_images': [16, 32],
  'whole_model_pickles': 2},
 'full_course_probe_verified': True,
 'full_dataset_pixels_and_cache_reverified': True,
 'state_roundtrips': [{'cell_index': 33,
   'strict_model_optimizer_gradient_rng_callback_state_restored': True,
   'actual_onecycle_curve_and_skips_verified': True,
   'cuda_rng_validation': 'Saved ByteTensor structure only; no CUDA generator replay claimed.'},
  {'cell_index': 47,
   'strict_model_optimizer_gradient_rng_callback_state_restored': True,
   'actual_onecycle_curve_and_skips_verified': True,
   'cuda_rng_validation': 'Saved ByteTensor structure only; no CUDA generator replay claimed.'},
  {'cell_index': 59,
   'strict_model_optimizer_gradient_rng_callback_state_restored': True,